# Natural Language Processing with Disaster Tweets
**Predict which Tweets are about real disasters and which ones are not**

***Acknowledgments***
This dataset was created by the company figure-eight and originally shared on their ‘Data For Everyone’ website here.

Tweet source: https://twitter.com/AnyOtherAnnaK/status/629195955506708480

Kaggle source: https://www.kaggle.com/competitions/nlp-getting-started/overview

In [131]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.metrics import accuracy_score, precision_score, f1_score
from sklearn.model_selection import train_test_split
from nltk import pos_tag
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/festusattornelson/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/festusattornelson/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/festusattornelson/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/festusattornelson/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

## Step 1. Data Exploration

In [132]:
df = pd.read_csv('/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/data/disaster_tweets_train.csv')

In [133]:
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [134]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        7613 non-null   int64
 1   keyword   7552 non-null   str  
 2   location  5080 non-null   str  
 3   text      7613 non-null   str  
 4   target    7613 non-null   int64
dtypes: int64(2), str(3)
memory usage: 297.5 KB


In [135]:
df.shape

(7613, 5)

In [136]:
df['keyword'] = df['keyword'].fillna('none')
df['location'] = df['location'].fillna('Unknown')

print("mising values after cleaning")
df.isnull().sum()

mising values after cleaning


id          0
keyword     0
location    0
text        0
target      0
dtype: int64

In [137]:
df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().apply(len)
df['hashtag_count'] = df['text'].str.count(r'#\w+')
df['mention_count'] = df['text'].str.count(r'@\w+')
df['url_count'] = df['text'].str.count(r'http\S+')

df[['text','char_count','word_count','hashtag_count','mention_count','url_count']].head()

,text,char_count,word_count,hashtag_count,mention_count,url_count
0,Our Deeds are the Reason of this #earthquake M...,69,13,1,0,0
1,Forest fire near La Ronge Sask. Canada,38,7,0,0,0
2,All residents asked to 'shelter in place' are ...,133,22,0,0,0
3,"13,000 people receive #wildfires evacuation or...",65,8,1,0,0
4,Just got sent this photo from Ruby #Alaska as ...,88,16,2,0,0


In [138]:
df_train =pd.DataFrame({'text':df['text'], 'target':df['target']})

In [139]:
df_train.head()

,text,target
0,Our Deeds are the Reason of this #earthquake M...,1
1,Forest fire near La Ronge Sask. Canada,1
2,All residents asked to 'shelter in place' are ...,1
3,"13,000 people receive #wildfires evacuation or...",1
4,Just got sent this photo from Ruby #Alaska as ...,1


In [140]:
df_train.target.value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

## Step 2. Preprocessing

In [141]:
# Lowercasing
df_train['text'] = df_train['text'].apply(lambda text: text.lower())
df_train.head()

,text,target
0,our deeds are the reason of this #earthquake m...,1
1,forest fire near la ronge sask. canada,1
2,all residents asked to 'shelter in place' are ...,1
3,"13,000 people receive #wildfires evacuation or...",1
4,just got sent this photo from ruby #alaska as ...,1


In [142]:
# Cleaning - remove numbers
df_train['text'] = df_train['text'].apply(lambda text: re.sub(r'\d+', '', text))
df_train.head()

,text,target
0,our deeds are the reason of this #earthquake m...,1
1,forest fire near la ronge sask. canada,1
2,all residents asked to 'shelter in place' are ...,1
3,", people receive #wildfires evacuation orders ...",1
4,just got sent this photo from ruby #alaska as ...,1


In [143]:
# Removing hashtags and special characters
df_train['text'] = df_train['text'].apply(lambda text: re.sub(r'[^a-zA-Z\s]', '', text))
df_train.head()

,text,target
0,our deeds are the reason of this earthquake ma...,1
1,forest fire near la ronge sask canada,1
2,all residents asked to shelter in place are be...,1
3,people receive wildfires evacuation orders in...,1
4,just got sent this photo from ruby alaska as s...,1


In [144]:
# Removing punctuation
df_train['text'] = df_train['text'].apply(lambda text: text.translate(str.maketrans('', '', string.punctuation)))
df_train.head()

,text,target
0,our deeds are the reason of this earthquake ma...,1
1,forest fire near la ronge sask canada,1
2,all residents asked to shelter in place are be...,1
3,people receive wildfires evacuation orders in...,1
4,just got sent this photo from ruby alaska as s...,1


In [145]:
# Tokenization
df_train['text'] = df_train['text'].apply(lambda text: text.split())
df_train.head()

,text,target
0,"[our, deeds, are, the, reason, of, this, earth...",1
1,"[forest, fire, near, la, ronge, sask, canada]",1
2,"[all, residents, asked, to, shelter, in, place...",1
3,"[people, receive, wildfires, evacuation, order...",1
4,"[just, got, sent, this, photo, from, ruby, ala...",1


In [146]:
# Filtering out non-alphabetic characters and short tokens
df_train['text'] = df_train['text'].apply(lambda text: [word for word in text if word.isalpha() and len(word) > 1   ])
df_train.head()

,text,target
0,"[our, deeds, are, the, reason, of, this, earth...",1
1,"[forest, fire, near, la, ronge, sask, canada]",1
2,"[all, residents, asked, to, shelter, in, place...",1
3,"[people, receive, wildfires, evacuation, order...",1
4,"[just, got, sent, this, photo, from, ruby, ala...",1


In [147]:
# Stop words removal
stop_words = set(stopwords.words('english'))
df_train['text'] = df_train['text'].apply(lambda text: [word for word in text if word not in stop_words])
df_train.head()

,text,target
0,"[deeds, reason, earthquake, may, allah, forgiv...",1
1,"[forest, fire, near, la, ronge, sask, canada]",1
2,"[residents, asked, shelter, place, notified, o...",1
3,"[people, receive, wildfires, evacuation, order...",1
4,"[got, sent, photo, ruby, alaska, smoke, wildfi...",1


In [148]:
# Lemmatization and stop word removal

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('N'):
        return 'n'
    elif tag.startswith('R'):
        return 'r'
    else:
        return 'n'

def tokenize(list_of_words):

  tagged_tokens = pos_tag(list_of_words)

  return [
      lemmatizer.lemmatize(word.lower(), get_wordnet_pos(tag))
      for word, tag in tagged_tokens]

df_train['text'] = df_train['text'].apply(tokenize)
print(df_train['text'].head())

0    [deed, reason, earthquake, may, allah, forgive...
1        [forest, fire, near, la, ronge, sask, canada]
2    [resident, ask, shelter, place, notify, office...
3    [people, receive, wildfire, evacuation, order,...
4    [get, sent, photo, ruby, alaska, smoke, wildfi...
Name: text, dtype: object


In [149]:
df_train['text'] = df_train['text'].apply(lambda x: ' '.join(x))

In [150]:
print(df_train['text'].head())

0           deed reason earthquake may allah forgive u
1                forest fire near la ronge sask canada
2    resident ask shelter place notify officer evac...
3    people receive wildfire evacuation order calif...
4    get sent photo ruby alaska smoke wildfires pou...
Name: text, dtype: str


## Step 3. Feature Extraction

### 3.1 Data Split

In [151]:
X = df['text']
y = df['target']

In [152]:
# Split into training and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, random_state=42, stratify=y)

In [153]:
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 6090
Test set size: 1523


### 3.2 Count and TF-IDF Vectorizers

In [154]:
# import vectorizer libraries
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [155]:
# Defining the vectorizers
vectorizers = {
    "CountVectorizer": CountVectorizer(
        max_features=5000,
        ngram_range=(1,2),
        stop_words='english'
    ),
    "TF-IDF": TfidfVectorizer(
        max_features=5000,
        ngram_range=(1,2),
        stop_words='english'
    )
}

In [156]:
for vec_name, vectorizer in vectorizers.items():
    print(f"Training {vec_name} completed")
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

Training CountVectorizer completed
Training TF-IDF completed


## Step 4. Model Training and Evaluation

In [157]:
# model libraries
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

In [158]:
# Defining models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Linear SVC": LinearSVC()
}

In [159]:
results = []

for model_name, model in models.items():
    print(f"Training {model_name} successful")

    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Vectorizer": vec_name,
        "Model": model_name,
        "Accuracy": accuracy,
        "F1_Score": f1
    })

Training Logistic Regression successful
Training Decision Tree successful
Training Random Forest successful
Training Linear SVC successful


In [160]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)

In [161]:
# Sorting the F1-Score
results_df = results_df.sort_values(
    by="F1_Score",
    ascending=False
)

### 4.1 Confusion Matrix for Baseline Models (Train vs. Test)

In [162]:
for vec_name, vectorizer in vectorizers.items():
    print(f"\n______ Vectorizer: {vec_name} ______")

    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    for model_name, model in models.items():
        print(f"\n______ Model: {model_name} ______")

        # Train the model
        model.fit(X_train_vec, y_train)

        # Predict on training set
        y_train_pred = model.predict(X_train_vec)
        cm_train = confusion_matrix(y_train, y_train_pred)
        print(f"Confusion Matrix (Train Set) for {model_name} with {vec_name}:\n{cm_train}")

        # Predict on test set
        y_test_pred = model.predict(X_test_vec)
        cm_test = confusion_matrix(y_test, y_test_pred)
        print(f"Confusion Matrix (Test Set) for {model_name} with {vec_name}:\n{cm_test}")


______ Vectorizer: CountVectorizer ______

______ Model: Logistic Regression ______
Confusion Matrix (Train Set) for Logistic Regression with CountVectorizer:
[[3356  117]
 [ 405 2212]]
Confusion Matrix (Test Set) for Logistic Regression with CountVectorizer:
[[762 107]
 [181 473]]

______ Model: Decision Tree ______
Confusion Matrix (Train Set) for Decision Tree with CountVectorizer:
[[3461   12]
 [  77 2540]]
Confusion Matrix (Test Set) for Decision Tree with CountVectorizer:
[[691 178]
 [205 449]]

______ Model: Random Forest ______
Confusion Matrix (Train Set) for Random Forest with CountVectorizer:
[[3451   22]
 [  67 2550]]
Confusion Matrix (Test Set) for Random Forest with CountVectorizer:
[[757 112]
 [219 435]]

______ Model: Linear SVC ______
Confusion Matrix (Train Set) for Linear SVC with CountVectorizer:
[[3413   60]
 [ 174 2443]]
Confusion Matrix (Test Set) for Linear SVC with CountVectorizer:
[[696 173]
 [181 473]]

______ Vectorizer: TF-IDF ______

______ Model: Logisti

In [163]:
print(results_df)

  Vectorizer                Model  Accuracy  F1_Score
0     TF-IDF  Logistic Regression  0.820092  0.772803
3     TF-IDF           Linear SVC  0.791202  0.748815
2     TF-IDF        Random Forest  0.783979  0.725146
1     TF-IDF        Decision Tree  0.731451  0.683681


In [164]:
# Selecting the Baseline model
best_baseline = results_df.iloc[0]

In [165]:
print(f"the best baseline model is {best_baseline}")

the best baseline model is Vectorizer                 TF-IDF
Model         Logistic Regression
Accuracy                 0.820092
F1_Score                 0.772803
Name: 0, dtype: object


## Step 5. Hyperparameter Tuning

### 5.1 GridSearch Hayperparameter Tuning

In [166]:
# Defining the Pipeline
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("model", LogisticRegression())
])

In [167]:
param_grid = [
    {
        # Logistic Regression + both vectorizers
        "vectorizer": [CountVectorizer(), TfidfVectorizer()],
        "vectorizer__max_features": [5000, 10000],
        "vectorizer__ngram_range": [(1,1), (1,2)],
        "model": [LogisticRegression(max_iter=1000)],
        "model__C": [0.01, 0.1, 1, 10]
    },
    {
        # Linear SVC (often best for text)
        "vectorizer": [CountVectorizer(), TfidfVectorizer()],
        "vectorizer__max_features": [5000, 10000],
        "vectorizer__ngram_range": [(1,1), (1,2)],
        "model": [LinearSVC()],
        "model__C": [0.01, 0.1, 1, 10]
    }
]

In [168]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [169]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 64 candidates, totalling 320 fits
[CV] END model=LogisticRegression(max_iter=1000), model__C=0.01, vectorizer=CountVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 1); total time=   0.1s
[CV] END model=LogisticRegression(max_iter=1000), model__C=0.01, vectorizer=CountVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 1); total time=   0.1s
[CV] END model=LogisticRegression(max_iter=1000), model__C=0.01, vectorizer=CountVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 1); total time=   0.1s
[CV] END model=LogisticRegression(max_iter=1000), model__C=0.01, vectorizer=CountVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 1); total time=   0.1s
[CV] END model=LogisticRegression(max_iter=1000), model__C=0.01, vectorizer=CountVectorizer(), vectorizer__max_features=10000, vectorizer__ngram_range=(1, 1); total time=   0.1s
[CV] END model=LogisticRegression(max_iter=1000), mo

/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END model=LinearSVC(), model__C=1, vectorizer=CountVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 2); total time=   0.3s
[CV] END model=LinearSVC(), model__C=1, vectorizer=CountVectorizer(), vectorizer__max_features=10000, vectorizer__ngram_range=(1, 1); total time=   0.2s
[CV] END model=LinearSVC(), model__C=1, vectorizer=CountVectorizer(), vectorizer__max_features=10000, vectorizer__ngram_range=(1, 1); total time=   0.2s
[CV] END model=LinearSVC(), model__C=1, vectorizer=CountVectorizer(), vectorizer__max_features=10000, vectorizer__ngram_range=(1, 1); total time=   0.3s
[CV] END model=LinearSVC(), model__C=1, vectorizer=CountVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 2); total time=   0.3s
[CV] END model=LinearSVC(), model__C=1, vectorizer=CountVectorizer(), vectorizer__max_features=10000, vectorizer__ngram_range=(1, 1); total time=   0.2s
[CV] END model=LinearSVC(), model__C=1, vectorizer=CountVectorizer(), vectorizer__ma

/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END model=LinearSVC(), model__C=1, vectorizer=TfidfVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 1); total time=   0.1s
[CV] END model=LinearSVC(), model__C=1, vectorizer=CountVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 2); total time=   0.4s
[CV] END model=LinearSVC(), model__C=1, vectorizer=TfidfVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 1); total time=   0.1s
[CV] END model=LinearSVC(), model__C=1, vectorizer=TfidfVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 1); total time=   0.1s
[CV] END model=LinearSVC(), model__C=1, vectorizer=CountVectorizer(), vectorizer__max_features=10000, vectorizer__ngram_range=(1, 2); total time=   0.4s
[CV] END model=LinearSVC(), model__C=1, vectorizer=TfidfVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 1); total time=   0.1s
[CV] END model=LinearSVC(), model__C=1, vectorizer=CountVectorizer(), vectorizer__max_f

...

/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END model=LinearSVC(), model__C=10, vectorizer=TfidfVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 2); total time=   0.4s
[CV] END model=LinearSVC(), model__C=10, vectorizer=TfidfVectorizer(), vectorizer__max_features=10000, vectorizer__ngram_range=(1, 1); total time=   0.2s
[CV] END model=LinearSVC(), model__C=10, vectorizer=TfidfVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 2); total time=   0.3s
[CV] END model=LinearSVC(), model__C=10, vectorizer=CountVectorizer(), vectorizer__max_features=10000, vectorizer__ngram_range=(1, 2); total time=   0.5s
[CV] END model=LinearSVC(), model__C=10, vectorizer=TfidfVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 2); total time=   0.3s
[CV] END model=LinearSVC(), model__C=10, vectorizer=TfidfVectorizer(), vectorizer__max_features=5000, vectorizer__ngram_range=(1, 2); total time=   0.3s
[CV] END model=LinearSVC(), model__C=10, vectorizer=TfidfVectorizer(), vectorize

/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END model=LinearSVC(), model__C=10, vectorizer=TfidfVectorizer(), vectorizer__max_features=10000, vectorizer__ngram_range=(1, 2); total time=   0.2s
[CV] END model=LinearSVC(), model__C=10, vectorizer=TfidfVectorizer(), vectorizer__max_features=10000, vectorizer__ngram_range=(1, 2); total time=   0.2s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model': [LogisticRegre...max_iter=1000)], 'model__C': [0.01, 0.1, ...], 'vectorizer': [CountVectorizer(), TfidfVectorizer()], 'vectorizer__max_features': [5000, 10000], ...}, {'model': [LinearSVC()], 'model__C': [0.01, 0.1, ...], 'vectorizer': [CountVectorizer(), TfidfVectorizer()], 'vectorizer__max_features': [5000, 10000], ...}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_pa

In [170]:
print(grid_search.best_params_)

print("\nBest CV F1 Score:")

print(grid_search.best_score_)

{'model': LogisticRegression(max_iter=1000), 'model__C': 10, 'vectorizer': TfidfVectorizer(), 'vectorizer__max_features': 10000, 'vectorizer__ngram_range': (1, 1)}

Best CV F1 Score:
0.7404655335031031


### 5.2 Comparison of Baseline and Tuned Model Performance

In [171]:
print("--- Best Baseline Model ---")
print(f"Vectorizer: {best_baseline['Vectorizer']}")
print(f"Model: {best_baseline['Model']}")
print(f"Accuracy: {best_baseline['Accuracy']:.4f}")
print(f"F1 Score: {best_baseline['F1_Score']:.4f}")

print("\n--- Hyperparameter Tuned Model ---")
print(f"Accuracy: {final_accuracy:.4f}")
print(f"F1 Score: {final_f1:.4f}")

--- Best Baseline Model ---
Vectorizer: TF-IDF
Model: Logistic Regression
Accuracy: 0.8201
F1 Score: 0.7728

--- Hyperparameter Tuned Model ---
Accuracy: 0.8089
F1 Score: 0.7742


### Step 6. Model Evaluation and Predictions

In [172]:
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

final_accuracy = accuracy_score(y_test, y_pred)
final_f1 = f1_score(y_test, y_pred)

In [173]:
print("Accuracy :", final_accuracy)
print("F1 Score :", final_f1)

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.8089297439264609
F1 Score : 0.7742435996896819

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.84      0.83       869
           1       0.79      0.76      0.77       654

    accuracy                           0.81      1523
   macro avg       0.81      0.80      0.80      1523
weighted avg       0.81      0.81      0.81      1523



In [174]:
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix")
print(cm)


Confusion Matrix
[[733 136]
 [155 499]]


## Step 7. Save Model

In [175]:
# Create models directory
from pathlib import Path
import joblib

model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)

joblib.dump(
    best_model,
    model_dir / "best_disaster_tweet_model.pkl"
)

print("\nModel saved successfully!")


Model saved successfully!


In [176]:
# Verify saved directory

from pathlib import Path
import joblib

model_path = Path("../models") / "best_disaster_tweet_model.pkl"

joblib.dump(best_model, model_path)

print(f"Model saved to:\n{model_path.resolve()}")

Model saved to:
/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/models/best_disaster_tweet_model.pkl


In [177]:
### Baseline model performed better than hyperparameter tuning